# 📍 Baseline Keypoint Detection Training

This notebook trains a CNN to detect baseline locations (y=0 line) in bar charts.

**Training data**: Generated synthetically using `generator.py` - no manual annotation required!

## Setup

In [ ]:
# Install dependencies
!pip install torchvision tqdm matplotlib numpy pillow -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
import csv
import os
import zipfile
from tqdm import tqdm
import matplotlib.pyplot as plt
from google.colab import files

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Upload Training Data

Upload the `training_data.zip` generated by `package_training_data.py`

In [ ]:
# Upload the training data ZIP
uploaded = files.upload()

# Extract
zip_filename = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_filename, 'r') as zf:
    zf.extractall('training_data')
    
print("Extracted files:")
!ls -la training_data/

In [ ]:
# Load keypoint training data
keypoint_data = []
with open('training_data/keypoint_training_data.csv', 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        keypoint_data.append({
            'image': row['image'],
            'orientation': row['orientation'],
            'baseline_coordinate': float(row['baseline_coordinate']),
            'axis_index': int(row['axis_index'])
        })
    
print(f"Loaded {len(keypoint_data)} keypoint samples")
print(f"Sample: {keypoint_data[0] if keypoint_data else 'empty'}")

## Data Loading

In [ ]:
class BaselineKeypointDataset(Dataset):
    """
    Dataset for baseline keypoint detection.
    
    Returns:
        - Image tensor (3, H, W)
        - Target: normalized y-coordinate of baseline [0, 1]
    """
    
    def __init__(self, data_list, image_dir, target_size=(256, 256), original_size=(600, 800)):
        self.data = data_list
        self.image_dir = image_dir
        self.target_size = target_size
        self.original_size = original_size
        
        self.transform = transforms.Compose([
            transforms.Resize(target_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        entry = self.data[idx]
        
        # Load image
        img_path = os.path.join(self.image_dir, entry['image'])
        try:
            img = Image.open(img_path).convert('RGB')
            orig_h, orig_w = img.size[1], img.size[0]
        except:
            # Fallback: create dummy image
            img = Image.new('RGB', self.target_size, color='white')
            orig_h, orig_w = self.original_size
        
        img_tensor = self.transform(img)
        
        # Normalize baseline coordinate to [0, 1]
        baseline_y = entry['baseline_coordinate']
        normalized_y = baseline_y / orig_h
        normalized_y = max(0, min(1, normalized_y))  # Clamp
        
        return img_tensor, torch.tensor([normalized_y], dtype=torch.float32)

In [ ]:
# Filter for vertical orientation only (most common)
vertical_data = [d for d in keypoint_data if d['orientation'] == 'vertical']
print(f"Vertical baseline samples: {len(vertical_data)}")

# Split data
n = len(vertical_data)
train_size = int(0.8 * n)
val_size = int(0.1 * n)

np.random.seed(42)
indices = np.random.permutation(n)
train_data = [vertical_data[i] for i in indices[:train_size]]
val_data = [vertical_data[i] for i in indices[train_size:train_size + val_size]]
test_data = [vertical_data[i] for i in indices[train_size + val_size:]]

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

In [ ]:
# Create datasets and loaders
IMAGE_DIR = 'training_data/images'
TARGET_SIZE = (224, 224)  # ResNet input size

train_dataset = BaselineKeypointDataset(train_data, IMAGE_DIR, target_size=TARGET_SIZE)
val_dataset = BaselineKeypointDataset(val_data, IMAGE_DIR, target_size=TARGET_SIZE)
test_dataset = BaselineKeypointDataset(test_data, IMAGE_DIR, target_size=TARGET_SIZE)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

## Model

In [ ]:
class BaselineRegressor(nn.Module):
    """
    CNN regressor for baseline Y-coordinate prediction.
    
    Uses ResNet18 backbone with regression head.
    Output: normalized Y coordinate [0, 1]
    """
    
    def __init__(self, pretrained=True):
        super().__init__()
        
        # ResNet18 backbone
        resnet = models.resnet18(pretrained=pretrained)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])  # Remove final FC
        
        # Regression head
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()  # Output in [0, 1]
        )
    
    def forward(self, x):
        features = self.backbone(x)
        return self.regressor(features)


model = BaselineRegressor(pretrained=True).to(device)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training

In [ ]:
# Training setup
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
criterion = nn.MSELoss()


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    total_samples = 0
    
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        
        optimizer.zero_grad()
        predictions = model(images)
        loss = criterion(predictions, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(images)
        total_samples += len(images)
    
    return total_loss / total_samples


@torch.no_grad()
def eval_epoch(model, loader, criterion, img_height=600):
    model.eval()
    total_loss = 0
    total_samples = 0
    total_pixel_error = 0
    
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        
        predictions = model(images)
        loss = criterion(predictions, targets)
        
        total_loss += loss.item() * len(images)
        
        # Pixel error (denormalize)
        pred_pixels = predictions.cpu().numpy() * img_height
        target_pixels = targets.cpu().numpy() * img_height
        total_pixel_error += np.abs(pred_pixels - target_pixels).sum()
        total_samples += len(images)
    
    avg_pixel_error = total_pixel_error / total_samples
    return total_loss / total_samples, avg_pixel_error

In [ ]:
# Training loop
NUM_EPOCHS = 30
best_val_error = float('inf')
train_losses, val_losses = [], []
val_errors = []

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_pixel_error = eval_epoch(model, val_loader, criterion)
    
    scheduler.step(val_loss)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_errors.append(val_pixel_error)
    
    # Save best model
    if val_pixel_error < best_val_error:
        best_val_error = val_pixel_error
        torch.save(model.state_dict(), 'best_keypoint_model.pt')
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
        print(f"  Train Loss: {train_loss:.6f}")
        print(f"  Val Loss: {val_loss:.6f}, Pixel Error: {val_pixel_error:.2f}px")

print(f"\nBest Val Pixel Error: {best_val_error:.2f}px")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label='Train')
axes[0].plot(val_losses, label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].set_title('Loss Curves')

axes[1].plot(val_errors, label='Val Pixel Error', color='orange')
axes[1].axhline(y=5, color='r', linestyle='--', label='5px threshold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Pixel Error')
axes[1].legend()
axes[1].set_title('Validation Pixel Error')

plt.tight_layout()
plt.show()

## Evaluation

In [ ]:
# Load best model and evaluate
model.load_state_dict(torch.load('best_keypoint_model.pt'))
test_loss, test_pixel_error = eval_epoch(model, test_loader, criterion)

print(f"Test Results:")
print(f"  MSE Loss: {test_loss:.6f}")
print(f"  Average Pixel Error: {test_pixel_error:.2f}px")
print(f"  (Compare to ~2-5px human annotation error)")

In [ ]:
# Visualize some predictions
@torch.no_grad()
def visualize_predictions(model, dataset, n=4):
    model.eval()
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    
    for i in range(n):
        idx = np.random.randint(len(dataset))
        img_tensor, target = dataset[idx]
        
        # Predict
        pred = model(img_tensor.unsqueeze(0).to(device)).cpu().item()
        target_val = target.item()
        
        # Denormalize image for display
        img_np = img_tensor.permute(1, 2, 0).numpy()
        img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_np = np.clip(img_np, 0, 1)
        
        h = img_np.shape[0]
        
        axes[i].imshow(img_np)
        axes[i].axhline(y=target_val * h, color='g', linewidth=2, label=f'GT: {target_val*600:.0f}px')
        axes[i].axhline(y=pred * h, color='r', linewidth=2, linestyle='--', label=f'Pred: {pred*600:.0f}px')
        axes[i].legend(fontsize=8)
        axes[i].set_title(f'Error: {abs(pred - target_val)*600:.1f}px')
    
    plt.tight_layout()
    plt.show()

visualize_predictions(model, test_dataset, n=4)

## Export Model

In [ ]:
# Save model for deployment
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'backbone': 'resnet18',
        'input_size': 224,
        'output': 'normalized_y'
    },
    'test_pixel_error': test_pixel_error,
    'test_mse_loss': test_loss
}, 'baseline_keypoint_cnn_final.pt')

# Download
files.download('baseline_keypoint_cnn_final.pt')
print("Model exported and downloaded!")

## Inference Helper

Code to use the trained model in your main codebase:

In [ ]:
inference_code = '''
# Add this to your baseline_detection.py

class TrainedBaselineDetector:
    """Use trained CNN to detect baseline location."""
    
    def __init__(self, model_path):
        checkpoint = torch.load(model_path, map_location='cpu')
        
        # Recreate model
        self.model = BaselineRegressor(pretrained=False)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()
        
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
    
    @torch.no_grad()
    def predict(self, image):
        """Predict baseline Y coordinate.
        
        Args:
            image: PIL Image or numpy array
        
        Returns:
            baseline_y: Y coordinate in pixels
        """
        if isinstance(image, np.ndarray):
            image = Image.fromarray(image)
        
        orig_h = image.size[1]
        img_tensor = self.transform(image).unsqueeze(0)
        
        normalized_y = self.model(img_tensor).item()
        return normalized_y * orig_h
'''
print(inference_code)